# Base de dados por setor censitário — Região Metropolitana de São Paulo (Censo 2022)

Este notebook seleciona os setores censitários dos 39 municípios da RMSP nos arquivos originais dos Agregados por Setores Censitários do Censo Demográfico 2022 (IBGE) e junta as tabelas Básico e Características dos Domicílios 1, 2 e 3 em uma única base.

**Entrada:** arquivos originais do IBGE (os mesmos usados no repositório `ad_cidadesp`): `basico.csv`, `domicilio1.csv`, `domicilio2.csv` e `domicilio3.csv`.
**Saída:** `setores_rmsp.csv` (uma linha por setor censitário da RMSP).

## 1. Configuração

Ajuste o caminho da pasta onde estão os arquivos do IBGE (no Colab, por exemplo, uma pasta do Google Drive).

In [ ]:
import pandas as pd
import numpy as np

PASTA = "data/"   # pasta com os arquivos originais do IBGE
ARQ_BASICO = PASTA + "basico.csv"
ARQ_DOMICILIOS = [PASTA + "domicilio1.csv", PASTA + "domicilio2.csv", PASTA + "domicilio3.csv"]
SEP, ENCODING = ";", "latin1"   # padrão dos arquivos do IBGE; ajuste se necessário

# Códigos IBGE dos 39 municípios da RMSP
RMSP = [
    3503901, 3505708, 3506607, 3509007, 3509205, 3510609, 3513009, 3513801, 3515004, 3515103,
    3515707, 3516309, 3516408, 3518305, 3518800, 3522208, 3522505, 3523107, 3525003, 3526209,
    3528502, 3529401, 3530607, 3534401, 3539103, 3539806, 3543303, 3544103, 3545001, 3546801,
    3547304, 3547809, 3548708, 3548807, 3549953, 3550308, 3552502, 3552809, 3556453,
]
assert len(RMSP) == 39

## 2. Tabela Básico: seleção dos setores da RMSP

A tabela Básico traz o código do município de cada setor. Ela define quais setores entram na base.

In [ ]:
basico = pd.read_csv(ARQ_BASICO, sep=SEP, encoding=ENCODING, dtype=str)
basico["CD_MUN"] = pd.to_numeric(basico["CD_MUN"], errors="coerce").astype("Int64")
basico = basico[basico["CD_MUN"].isin(RMSP)].copy()

print(f"Setores da RMSP: {len(basico):,}")
print(f"Municípios encontrados: {basico['CD_MUN'].nunique()} de 39")
setores_rmsp = set(basico["CD_SETOR"])

## 3. Tabelas de Características dos Domicílios

Os arquivos são lidos em partes (*chunks*), mantendo apenas os setores da RMSP, para não sobrecarregar a memória.

In [ ]:
def ler_setores(arquivo, setores, tamanho=200_000):
    partes = []
    for chunk in pd.read_csv(arquivo, sep=SEP, encoding=ENCODING, dtype=str, chunksize=tamanho):
        partes.append(chunk[chunk["CD_SETOR"].isin(setores)])
    return pd.concat(partes, ignore_index=True)

domicilios = []
for arq in ARQ_DOMICILIOS:
    d = ler_setores(arq, setores_rmsp)
    # remove colunas de identificação repetidas (ficam só na tabela Básico)
    d = d[["CD_SETOR"] + [c for c in d.columns if c.upper().startswith("V")]]
    print(f"{arq}: {len(d):,} setores | {d.shape[1] - 1} variáveis")
    domicilios.append(d)

## 4. Junção das tabelas

In [ ]:
base = basico.copy()
for d in domicilios:
    base = base.merge(d, on="CD_SETOR", how="left")
print(f"Base consolidada: {base.shape[0]:,} setores e {base.shape[1]} colunas")

## 5. Conversão numérica e valores omitidos por sigilo

Nos Agregados por Setores, o IBGE omite os dados de setores com poucos domicílios, por sigilo estatístico, marcando-os com `X`. Esses valores são convertidos em ausentes (`NaN`), e não em zero, porque o valor real é desconhecido. A vírgula decimal é convertida em ponto.

In [ ]:
id_cols = [c for c in basico.columns if not c.lower().startswith("v")]
var_cols = [c for c in base.columns if c not in id_cols]

base[var_cols] = (base[var_cols]
                  .replace({"X": np.nan, "x": np.nan, ".": np.nan, "": np.nan})
                  .apply(lambda s: pd.to_numeric(s.astype(str).str.replace(",", "."), errors="coerce")))

ausentes = base[var_cols].isna().all(axis=1).sum()
print(f"Setores com todos os valores omitidos (sigilo ou sem domicílios): {ausentes:,}")
print(f"Porcentagem média de valores ausentes por variável: {base[var_cols].isna().mean().mean() * 100:.1f}%")

## 6. Salvar a base

In [ ]:
base.to_csv("setores_rmsp.csv", index=False, sep=";", encoding="utf-8")
print("setores_rmsp.csv salvo.")